# FastCheckAI - Pipeline de Comparação de PDFs

Este notebook implementa o pipeline completo de comparação de documentos técnicos em PDF usando processamento híbrido e análise semântica.

## Setup e Validação

Antes de começar, vamos validar que todas as dependências estão instaladas e o ambiente está configurado corretamente.

### 1. Validação de Imports

In [ ]:
# Validação de imports essenciais
import sys
import pymupdf
import pdfplumber
import pandas as pd
import agno
from openai import OpenAI

print("✅ Todos os imports essenciais foram bem-sucedidos!")
print(f"Python version: {sys.version}")
print(f"PyMuPDF version: {pymupdf.__version__}")
print(f"pdfplumber version: {pdfplumber.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"agno version: {agno.__version__}")

### 2. Validação de Configuração (API Keys)

In [ ]:
# Validação de configuração e API keys
from src.config import (
    OPENAI_API_KEY,
    MODEL,
    TEMPERATURE,
    FUZZY_MATCH_THRESHOLD,
    SEMANTIC_CONFIDENCE_THRESHOLD,
    logger
)

print("✅ Configuração carregada com sucesso!")
print(f"Modelo: {MODEL}")
print(f"Temperature: {TEMPERATURE}")
print(f"Fuzzy Match Threshold: {FUZZY_MATCH_THRESHOLD}")
print(f"Semantic Confidence Threshold: {SEMANTIC_CONFIDENCE_THRESHOLD}")
print(f"API Key configurada: {'Sim' if OPENAI_API_KEY else 'Não'}")

### 3. Teste de Conexão OpenAI API

In [ ]:
# Teste de conexão com OpenAI API
from openai import OpenAI
import time

client = OpenAI(api_key=OPENAI_API_KEY)

start_time = time.time()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one word"}],
    temperature=TEMPERATURE
)
elapsed_time = time.time() - start_time

print(f"✅ Conexão OpenAI API bem-sucedida!")
print(f"Resposta: {response.choices[0].message.content}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")
print(f"Tokens usados: {response.usage.total_tokens}")

### 4. Validação Agno Framework

O Agno Framework será usado para orquestração de agentes LLM para análise semântica. Vamos validar que funciona corretamente.

In [ ]:
# Teste Agno Framework - Hello World
from agno import Agent
import time

# Criar agente Agno com GPT-4o
agent = Agent(
    model=MODEL,
    instructions="You are a helpful assistant specialized in technical documentation.",
)

# Teste simples
start_time = time.time()
response = agent.run("Explain what a technical standard is in one sentence")
elapsed_time = time.time() - start_time

print("✅ Agno Framework validado com sucesso!")
print(f"Resposta do agente: {response}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")

---

## ✅ Setup Completo!

Se todas as células acima foram executadas sem erros, seu ambiente está pronto para desenvolvimento.

**Próximos passos:**
1. Implementar pipeline de extração de PDF (Feature 2)
2. Desenvolver sistema de alinhamento heurístico (Feature 3)
3. Integrar análise semântica com Agno (Feature 4)

---

## 1. Carregamento e Validação de PDFs

Esta seção implementa o carregamento robusto dos dois PDFs a serem comparados com:
- Validação de tamanho (≤25MB)
- Detecção automática de tipo (nativo vs escaneado)
- Tratamento de erros claros

### 1.1 Carregamento dos PDFs

In [ ]:
# Importar módulo de carregamento de PDFs
from src.pdf_loader import load_pdf, detect_pdf_type

# Carregar os dois PDFs para comparação
print("📄 Carregando PDFs...")
pdf_a = load_pdf("data/inputs/doc_a.pdf")
pdf_b = load_pdf("data/inputs/doc_b.pdf")

print(f"\n✅ PDF A carregado: {pdf_a.page_count} páginas")
print(f"✅ PDF B carregado: {pdf_b.page_count} páginas")

### 1.2 Detecção de Tipo de PDF

In [ ]:
# Detectar tipo dos PDFs (nativo vs escaneado)
print("🔍 Detectando tipo dos PDFs...\n")

type_a = detect_pdf_type(pdf_a)
type_b = detect_pdf_type(pdf_b)

print(f"PDF A: {'✅ Nativo (texto extraível)' if type_a['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_a['char_count']} caracteres na página 1")

print(f"\nPDF B: {'✅ Nativo (texto extraível)' if type_b['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_b['char_count']} caracteres na página 1")

# Avisar se algum PDF requer OCR
if type_a['requires_ocr'] or type_b['requires_ocr']:
    print("\n⚠️  ATENÇÃO: Um ou mais PDFs parecem ser escaneados e podem requerer OCR para extração completa.")

### 1.3 Testes de Validação

Vamos testar os casos de erro para garantir que as validações estão funcionando corretamente.

In [ ]:
# Teste 1: Validação de tamanho (PDF muito grande)
print("🧪 Teste 1: Validação de tamanho máximo\n")
try:
    large_pdf = load_pdf("data/inputs/large_file.pdf")
    print("❌ FALHA: PDF grande deveria ter sido rejeitado")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 2: PDF corrompido
print("🧪 Teste 2: Tratamento de PDF corrompido\n")
try:
    corrupt_pdf = load_pdf("data/inputs/corrupted.pdf")
    print("❌ FALHA: PDF corrompido deveria ter gerado erro")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 3: Arquivo inexistente
print("🧪 Teste 3: Tratamento de arquivo inexistente\n")
try:
    missing_pdf = load_pdf("data/inputs/nao_existe.pdf")
    print("❌ FALHA: Arquivo inexistente deveria gerar erro")
except FileNotFoundError as e:
    print(f"✅ SUCESSO: FileNotFoundError: {e}\n")

# Teste 4: PDF escaneado (sem texto)
print("🧪 Teste 4: Detecção de PDF escaneado\n")
try:
    scanned_pdf = load_pdf("data/inputs/scanned_sample.pdf")
    scanned_type = detect_pdf_type(scanned_pdf)
    if scanned_type['requires_ocr']:
        print(f"✅ SUCESSO: PDF detectado como escaneado ({scanned_type['char_count']} chars)")
    else:
        print(f"❌ FALHA: PDF deveria ser detectado como escaneado")
    scanned_pdf.close()
except Exception as e:
    print(f"❌ ERRO: {e}")

print("\n✅ Todos os testes de validação concluídos!")